# 03 - Temporal aggregation on Google Earth Engine with export to Google Drive

This notebook reads the uploaded 3-bit shadow rasters from an Earth Engine image collection, aggregates them over time, and exports compact GeoTIFF bundles to Google Drive.


## 1. Install dependencies

Run this only if the current environment does not already include the Earth Engine Python API.


In [ ]:
# %pip install earthengine-api


## 2. Import the libraries used in the aggregation stage

The aggregation logic is implemented with the Earth Engine Python API and a small configuration dataclass.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Optional

import ee


## 3. Initialize Earth Engine

Set your project in the configuration cell below, then initialize the Earth Engine session.


In [ ]:
def initialize_earth_engine(gee_project: str, force_auth: bool = False) -> None:
    if not gee_project or "YOUR_" in gee_project:
        raise ValueError("Set a valid value in cfg.gee_project.")

    if force_auth:
        ee.Authenticate()

    ee.Initialize(project=gee_project)
    print(f"Earth Engine initialized on project: {gee_project}")


## 4. Configure the aggregation parameters

This block defines:
- the source image collection
- the output Drive folder
- the target `height_tag`
- the timezone used for local hour derivation
- the time bands
- export grid options
- the NoData value to be embedded in the GeoTIFF exports


In [ ]:
@dataclass
class Config:
    gee_project: str = "YOUR_GEE_PROJECT"
    collection_asset: str = "projects/YOUR_GEE_PROJECT/assets/shadow_input_collection"
    drive_folder: str = "shadow_aggregated"

    height_tag: str = "h1p5m"
    timezone_name: str = "Europe/Rome"
    step_minutes: int = 15
    nodata: int = 255

    # Export counts as uint16 when safe, otherwise uint32.
    count_dtype: str = "auto"   # allowed: "auto", "uint16", "uint32"

    time_bands: dict[str, tuple[int, int]] | None = None

    use_source_grid: bool = True
    export_scale: float = 0.5
    export_crs: str = "EPSG:32632"
    max_pixels: int = int(2e9)

    fail_on_impossible_values: bool = True
    export_cloud_optimized: bool = False

    export_static_masks: bool = True
    export_global_core_stats: bool = True
    export_global_hourly_core_stats: bool = True
    export_global_timeband_core_stats: bool = True
    export_monthly_bundles: bool = True

    def __post_init__(self):
        if self.time_bands is None:
            self.time_bands = {
                "earlymorning": (6, 9),
                "morning": (9, 12),
                "peakthermal": (12, 15),
                "afternoon": (15, 18),
                "evening": (18, 21),
            }

cfg = Config()
cfg


## 5. Define the 3-bit decoding logic and local time metadata

The helper functions below:
- identify valid pixels
- decode `G_h`, `S_h`, and `M_h`
- build static masks
- add local time properties from `system:time_start`


In [ ]:
def _valid(image: ee.Image, nodata: int = 255) -> ee.Image:
    return image.neq(nodata)


def _bit0_G(image: ee.Image) -> ee.Image:
    return image.bitwiseAnd(1).neq(0)


def _bit1_S(image: ee.Image) -> ee.Image:
    return image.bitwiseAnd(2).neq(0)


def _bit2_M(image: ee.Image) -> ee.Image:
    return image.bitwiseAnd(4).neq(0)


def make_structure_mask(image: ee.Image, nodata: int = 255) -> ee.Image:
    valid = _valid(image, nodata)
    return _bit2_M(image).updateMask(valid).rename("structure_mask")


def make_open_ground_mask(image: ee.Image, nodata: int = 255) -> ee.Image:
    valid = _valid(image, nodata)
    return _bit2_M(image).Not().updateMask(valid).rename("open_ground_mask")


def make_impossible_mask(image: ee.Image, nodata: int = 255) -> ee.Image:
    valid = _valid(image, nodata)
    impossible = image.eq(2).Or(image.eq(3))
    return impossible.updateMask(valid).rename("impossible")


def add_local_time_properties(collection: ee.ImageCollection, tz_name: str) -> ee.ImageCollection:
    def add_props(img: ee.Image) -> ee.Image:
        date = ee.Date(img.get("system:time_start"))
        return img.set({
            "local_year": ee.Number.parse(date.format("YYYY", tz_name)),
            "local_month": ee.Number.parse(date.format("M", tz_name)),
            "local_hour": ee.Number.parse(date.format("H", tz_name)),
            "local_day_key": date.format("YYYY-MM-dd", tz_name),
            "local_month_key": date.format("YYYYMM", tz_name),
            "timezone_name": tz_name,
        })
    return collection.map(add_props)


## 6. Define the temporal filters and the compact aggregation bundles

These helpers build:
- full-period core stats
- hourly bundles
- time-band bundles
- monthly bundles


In [ ]:
def filter_by_month(collection: ee.ImageCollection, year: int, month: int) -> ee.ImageCollection:
    return collection.filter(
        ee.Filter.And(
            ee.Filter.eq("local_year", year),
            ee.Filter.eq("local_month", month),
        )
    )


def filter_by_hour(collection: ee.ImageCollection, hour: int) -> ee.ImageCollection:
    return collection.filter(ee.Filter.eq("local_hour", hour))


def filter_by_hour_range(collection: ee.ImageCollection, h_start: int, h_end: int) -> ee.ImageCollection:
    return collection.filter(
        ee.Filter.And(
            ee.Filter.gte("local_hour", h_start),
            ee.Filter.lt("local_hour", h_end),
        )
    )


def get_distinct_months(collection: ee.ImageCollection) -> list[tuple[int, int]]:
    month_keys = collection.aggregate_array("local_month_key").distinct().sort().getInfo()
    return [
        (int(str(key)[:4]), int(str(key)[4:6]))
        for key in month_keys
        if key is not None
    ]


def get_distinct_hours(collection: ee.ImageCollection) -> list[int]:
    hours = collection.aggregate_array("local_hour").distinct().sort().getInfo()
    return [int(h) for h in hours if h is not None]


def count_distinct_days(collection: ee.ImageCollection) -> int:
    return int(collection.aggregate_array("local_day_key").distinct().size().getInfo())


def choose_count_dtype(n_images: int, requested: str = "auto") -> str:
    requested = requested.lower()
    if requested not in {"auto", "uint16", "uint32"}:
        raise ValueError("count_dtype must be 'auto', 'uint16', or 'uint32'.")

    if requested == "auto":
        return "uint16" if n_images <= 65535 else "uint32"
    return requested


def cast_count_image(image: ee.Image, count_dtype: str) -> ee.Image:
    if count_dtype == "uint16":
        return image.toUint16()
    if count_dtype == "uint32":
        return image.toUint32()
    raise ValueError(f"Unsupported count_dtype: {count_dtype}")


def compute_core_stats(
    collection: ee.ImageCollection,
    count_dtype: str,
    nodata: int = 255,
    prefix: str = "",
) -> ee.Image:
    valid_count = collection.map(
        lambda img: ee.Image.constant(1).updateMask(_valid(img, nodata)).rename("valid_count")
    ).sum().rename(f"{prefix}valid_count")

    count_g = collection.map(
        lambda img: _bit0_G(img).updateMask(_valid(img, nodata)).rename("count_G")
    ).sum().rename(f"{prefix}count_G")

    count_s = collection.map(
        lambda img: _bit1_S(img).updateMask(_valid(img, nodata)).rename("count_S")
    ).sum().rename(f"{prefix}count_S")

    valid_count = cast_count_image(valid_count, count_dtype)
    count_g = cast_count_image(count_g, count_dtype)
    count_s = cast_count_image(count_s, count_dtype)

    mask = valid_count.gt(0)
    return ee.Image.cat([valid_count, count_g, count_s]).updateMask(mask)


def make_static_masks_bundle(first_image: ee.Image, nodata: int = 255) -> ee.Image:
    return ee.Image.cat([
        make_structure_mask(first_image, nodata),
        make_open_ground_mask(first_image, nodata),
    ])


def build_hourly_bundle(collection: ee.ImageCollection, hours: list[int], count_dtype: str, nodata: int = 255) -> ee.Image:
    images = [compute_core_stats(filter_by_hour(collection, hour), count_dtype, nodata, prefix=f"h{hour:02d}_") for hour in hours]
    if not images:
        return ee.Image.constant(0).rename("empty").updateMask(ee.Image.constant(0))
    return ee.Image.cat(images)


def build_timeband_bundle(
    collection: ee.ImageCollection,
    hours_present: list[int],
    time_bands: dict[str, tuple[int, int]],
    count_dtype: str,
    nodata: int = 255,
) -> ee.Image:
    images = []
    for band_name, (h_start, h_end) in time_bands.items():
        if not any(h_start <= hour < h_end for hour in hours_present):
            continue
        b_col = filter_by_hour_range(collection, h_start, h_end)
        images.append(compute_core_stats(b_col, count_dtype, nodata, prefix=f"{band_name}_"))
    if not images:
        return ee.Image.constant(0).rename("empty").updateMask(ee.Image.constant(0))
    return ee.Image.cat(images)


def build_monthly_bundle(
    month_col: ee.ImageCollection,
    month_hours: list[int],
    cfg: Config,
    count_dtype: str,
) -> ee.Image:
    images = [compute_core_stats(month_col, count_dtype, cfg.nodata, prefix="month_")]

    for hour in month_hours:
        h_col = filter_by_hour(month_col, hour)
        images.append(compute_core_stats(h_col, count_dtype, cfg.nodata, prefix=f"h{hour:02d}_"))

    for band_name, (h_start, h_end) in cfg.time_bands.items():
        if not any(h_start <= hour < h_end for hour in month_hours):
            continue
        b_col = filter_by_hour_range(month_col, h_start, h_end)
        images.append(compute_core_stats(b_col, count_dtype, cfg.nodata, prefix=f"{band_name}_"))

    return ee.Image.cat(images)


## 7. Define the export grid, anomaly check and GeoTIFF export helper

This version explicitly sets:
- the output `nodata` value
- optional Cloud Optimized GeoTIFF export
- the source grid or a custom export grid


In [ ]:
def get_export_grid(first_image: ee.Image, cfg: Config) -> dict:
    if not cfg.use_source_grid:
        return {
            "crs": cfg.export_crs,
            "crsTransform": None,
            "scale": cfg.export_scale,
        }

    proj_info = ee.Image(first_image).projection().getInfo()
    crs = proj_info.get("crs") or cfg.export_crs
    crs_transform = proj_info.get("transform")

    return {
        "crs": crs,
        "crsTransform": crs_transform,
        "scale": None if crs_transform is not None else cfg.export_scale,
    }


def count_impossible_pixels(
    collection: ee.ImageCollection,
    region: ee.Geometry,
    export_grid: dict,
    nodata: int,
    max_pixels: int,
) -> int:
    impossible_sum = collection.map(lambda img: make_impossible_mask(img, nodata)).sum().rename("impossible")

    reduce_kwargs = {
        "reducer": ee.Reducer.sum(),
        "geometry": region,
        "maxPixels": max_pixels,
    }
    if export_grid.get("crs"):
        reduce_kwargs["crs"] = export_grid["crs"]
    if export_grid.get("crsTransform") is not None:
        reduce_kwargs["crsTransform"] = export_grid["crsTransform"]
    elif export_grid.get("scale") is not None:
        reduce_kwargs["scale"] = export_grid["scale"]

    value = impossible_sum.reduceRegion(**reduce_kwargs).get("impossible").getInfo()
    return int(value or 0)


def export_image_to_drive(
    image: ee.Image,
    name: str,
    cfg: Config,
    region: ee.Geometry,
    export_grid: dict,
    extra_properties: dict | None = None,
) -> ee.batch.Task:
    if extra_properties:
        image = image.set(extra_properties)

    export_image = image.unmask(cfg.nodata)

    kwargs = {
        "image": export_image,
        "description": name,
        "folder": cfg.drive_folder,
        "fileNamePrefix": name,
        "fileFormat": "GeoTIFF",
        "formatOptions": {
            "noData": cfg.nodata,
            "cloudOptimized": cfg.export_cloud_optimized,
        },
        "region": region,
        "maxPixels": cfg.max_pixels,
    }

    if export_grid.get("crs"):
        kwargs["crs"] = export_grid["crs"]
    if export_grid.get("crsTransform") is not None:
        kwargs["crsTransform"] = export_grid["crsTransform"]
    elif export_grid.get("scale") is not None:
        kwargs["scale"] = export_grid["scale"]

    task = ee.batch.Export.image.toDrive(**kwargs)
    task.start()
    return task


def get_task_statuses(tasks: list[ee.batch.Task]) -> list[dict]:
    out = []
    for task in tasks:
        status = task.status()
        out.append({
            "id": status.get("id"),
            "state": status.get("state"),
            "description": status.get("description"),
            "error_message": status.get("error_message"),
        })
    return out


## 8. Preview the source collection and the derived local time metadata

This cell is useful before launching exports because it confirms:
- how many images are available after filtering by `height_tag`
- which months and hours are present
- how many distinct local days are represented


In [ ]:
# Example:
# initialize_earth_engine(cfg.gee_project, force_auth=False)
#
# collection_preview = ee.ImageCollection(cfg.collection_asset).filter(
#     ee.Filter.eq("height_tag", cfg.height_tag)
# )
# collection_preview = add_local_time_properties(collection_preview, cfg.timezone_name)
#
# print("Images found:", collection_preview.size().getInfo())
# print("Distinct hours:", get_distinct_hours(collection_preview))
# print("Distinct months:", get_distinct_months(collection_preview))
# print("Distinct days:", count_distinct_days(collection_preview))


## 9. Run the temporal aggregation and submit the exports

This block executes the full aggregation workflow and submits one Drive export task for each selected output family.


In [ ]:
def run_aggregation_and_export(cfg: Config) -> list[ee.batch.Task]:
    print("Compact shadow aggregation on Google Earth Engine")
    print(f"  Collection asset: {cfg.collection_asset}")
    print(f"  Height tag:       {cfg.height_tag}")
    print(f"  Timezone:         {cfg.timezone_name}")
    print(f"  Drive folder:     {cfg.drive_folder}")

    full_collection = ee.ImageCollection(cfg.collection_asset)
    collection = full_collection.filter(ee.Filter.eq("height_tag", cfg.height_tag))
    collection = add_local_time_properties(collection, cfg.timezone_name)

    n_images = collection.size().getInfo()
    print(f"  Images after height filter: {n_images}")
    if n_images == 0:
        raise RuntimeError(
            "No images were found after filtering by height_tag. "
            "Check cfg.collection_asset and cfg.height_tag."
        )

    count_dtype = choose_count_dtype(n_images, cfg.count_dtype)
    print(f"  Count dtype:      {count_dtype}")

    region = collection.geometry().bounds()
    first_image = ee.Image(collection.first())
    export_grid = get_export_grid(first_image, cfg)

    print(f"  Export CRS: {export_grid['crs']}")
    if export_grid.get("crsTransform") is not None:
        print(f"  Export crsTransform: {export_grid['crsTransform']}")
    else:
        print(f"  Export scale: {export_grid['scale']}")
    print(f"  Export nodata: {cfg.nodata}")

    if cfg.fail_on_impossible_values:
        impossible_pixels = count_impossible_pixels(
            collection=collection,
            region=region,
            export_grid=export_grid,
            nodata=cfg.nodata,
            max_pixels=cfg.max_pixels,
        )
        print(f"  Impossible values (2/3): {impossible_pixels}")
        if impossible_pixels > 0:
            raise RuntimeError(
                "Impossible values (2 or 3) were found in the collection. "
                "Fix the source rasters before aggregation."
            )

    months = get_distinct_months(collection)
    hours = get_distinct_hours(collection)
    total_n_days = count_distinct_days(collection)

    print(f"  Months found: {months}")
    print(f"  Hours found:  {hours}")
    print(f"  Distinct days: {total_n_days}")

    base_props = {
        "height_tag": cfg.height_tag,
        "timezone_name": cfg.timezone_name,
        "step_minutes": cfg.step_minutes,
        "encoding": "bit0=G_h, bit1=S_h, bit2=M_h",
        "compact_export": True,
        "count_dtype": count_dtype,
        "nodata_value": cfg.nodata,
    }

    tasks: list[ee.batch.Task] = []

    if cfg.export_static_masks:
        tasks.append(export_image_to_drive(
            make_static_masks_bundle(first_image, cfg.nodata),
            "static_masks",
            cfg,
            region,
            export_grid,
            {**base_props, "product_family": "static_masks"},
        ))

    if cfg.export_global_core_stats:
        tasks.append(export_image_to_drive(
            compute_core_stats(collection, count_dtype, cfg.nodata),
            "global_core_stats",
            cfg,
            region,
            export_grid,
            {
                **base_props,
                "product_family": "global_core_stats",
                "n_days": total_n_days,
                "n_images": n_images,
            },
        ))

    if cfg.export_global_hourly_core_stats:
        tasks.append(export_image_to_drive(
            build_hourly_bundle(collection, hours, count_dtype, cfg.nodata),
            "global_hourly_core_stats",
            cfg,
            region,
            export_grid,
            {
                **base_props,
                "product_family": "global_hourly_core_stats",
                "n_days": total_n_days,
                "hours": ",".join(f"{h:02d}" for h in hours),
            },
        ))

    if cfg.export_global_timeband_core_stats:
        tasks.append(export_image_to_drive(
            build_timeband_bundle(collection, hours, cfg.time_bands, count_dtype, cfg.nodata),
            "global_timeband_core_stats",
            cfg,
            region,
            export_grid,
            {
                **base_props,
                "product_family": "global_timeband_core_stats",
                "n_days": total_n_days,
                "time_bands": ",".join(cfg.time_bands.keys()),
            },
        ))

    if cfg.export_monthly_bundles:
        for year, month in months:
            month_col = filter_by_month(collection, year, month)
            month_id = f"{year}{month:02d}"
            n_days_month = count_distinct_days(month_col)
            month_hours = get_distinct_hours(month_col)

            tasks.append(export_image_to_drive(
                build_monthly_bundle(month_col, month_hours, cfg, count_dtype),
                f"monthly_bundle_{month_id}",
                cfg,
                region,
                export_grid,
                {
                    **base_props,
                    "product_family": "monthly_bundle",
                    "year": year,
                    "month": month,
                    "month_key": month_id,
                    "n_days": n_days_month,
                    "hours": ",".join(f"{h:02d}" for h in month_hours),
                    "time_bands": ",".join(cfg.time_bands.keys()),
                },
            ))

    print(f"\nSubmitted tasks: {len(tasks)}")
    print(f"Drive folder: {cfg.drive_folder}")
    return tasks


## 10. Launch the exports

Uncomment and run the sequence below after you have configured the notebook and confirmed the source collection.


In [ ]:
# initialize_earth_engine(cfg.gee_project, force_auth=False)
# tasks = run_aggregation_and_export(cfg)
# print(f"Tasks submitted: {len(tasks)}")


## 11. Check the task status after submission

Use this block after the export tasks have been created.


In [ ]:
# statuses = get_task_statuses(tasks)
# for row in statuses:
#     print(f"{row['description']}: {row['state']}")
#     if row['error_message']:
#         print("  error:", row['error_message'])


## 12. How to interpret the exported files

- `static_masks` contains the static structure and open-ground masks
- `global_core_stats` contains `valid_count`, `count_G`, and `count_S` for the full period
- `global_hourly_core_stats` groups the same counts by local hour
- `global_timeband_core_stats` groups the same counts by configured time bands
- `monthly_bundle_YYYYMM` contains the monthly total plus hourly and time-band detail for that month

Downstream frequencies are derived later as:
- `freq_G = count_G / valid_count`
- `freq_S = count_S / valid_count`


## 13. Troubleshooting

- If the collection is empty after filtering by `height_tag`, validate the upload metadata first.
- If the export grid looks wrong, disable `use_source_grid` and set `export_crs` / `export_scale`.
- If impossible values are reported, inspect the extraction outputs before aggregating.
